In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

(x_train, _), (x_test, _) = tf.keras.datasets.fashion_mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

input_layer = layers.Input(shape=(784,))

encoded = layers.Dense(128, activation="relu")(input_layer)
encoded = layers.Dense(64, activation="relu")(encoded)

decoded = layers.Dense(128, activation="relu")(encoded)
decoded = layers.Dense(784, activation="sigmoid")(decoded)

autoencoder = models.Model(input_layer, decoded)

autoencoder.compile(
    optimizer="adam",
    loss="mse"
)

autoencoder.fit(
    x_train,
    x_train,
    epochs=5,
    batch_size=256,
    validation_data=(x_test, x_test)
)

reconstructed = autoencoder.predict(x_test[:1])

plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(x_test[0].reshape(28, 28), cmap="gray")
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(reconstructed[0].reshape(28, 28), cmap="gray")
plt.title("Reconstructed")
plt.axis("off")

plt.show()

In [ ]:
# Task 2

autoencoder.compile(
    optimizer="adam",
    loss="mse"
)

autoencoder.compile(
    optimizer="adam",
    loss="binary_crossentropy"
)

In [ ]:
# Task 3

import numpy as np
import matplotlib.pyplot as plt

images = x_test[:10]

noise = np.random.normal(
    0,
    0.3,
    images.shape
)

noisy_images = images + noise
noisy_images = np.clip(noisy_images, 0, 1)

denoised_images = autoencoder.predict(noisy_images)

plt.figure(figsize=(15, 5))

for i in range(10):
    plt.subplot(2, 10, i + 1)
    plt.imshow(noisy_images[i].reshape(28, 28), cmap="gray")
    plt.title("Noisy")
    plt.axis("off")

    plt.subplot(2, 10, i + 11)
    plt.imshow(denoised_images[i].reshape(28, 28), cmap="gray")
    plt.title("Denoised")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Task 4

import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

(x_train, _), (x_test, _) = tf.keras.datasets.fashion_mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

latent_dim = 2

inputs = layers.Input(shape=(784,))

h = layers.Dense(256, activation="relu")(inputs)

z_mean = layers.Dense(latent_dim)(h)
z_log_var = layers.Dense(latent_dim)(h)

def sampling(args):
    mean, log_var = args
    epsilon = tf.random.normal(shape=tf.shape(mean))
    return mean + tf.exp(0.5 * log_var) * epsilon

z = layers.Lambda(sampling)([z_mean, z_log_var])

encoder = tf.keras.Model(
    inputs,
    [z_mean, z_log_var, z]
)

latent_inputs = layers.Input(shape=(latent_dim,))

x = layers.Dense(256, activation="relu")(latent_inputs)
outputs = layers.Dense(784, activation="sigmoid")(x)

decoder = tf.keras.Model(
    latent_inputs,
    outputs
)

class VAE(tf.keras.Model):

    def train_step(self, data):

        with tf.GradientTape() as tape:

            mean, log_var, z = encoder(data)

            reconstruction = decoder(z)

            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(
                    tf.keras.losses.binary_crossentropy(
                        data,
                        reconstruction
                    ),
                    axis=-1
                )
            )

            kl_loss = -0.5 * tf.reduce_mean(
                1 + log_var
                - tf.square(mean)
                - tf.exp(log_var)
            )

            total_loss = reconstruction_loss + kl_loss

        gradients = tape.gradient(
            total_loss,
            self.trainable_weights
        )

        self.optimizer.apply_gradients(
            zip(gradients, self.trainable_weights)
        )

        return {
            "loss": total_loss
        }

vae = VAE()

vae.compile(
    optimizer="adam"
)

vae.fit(
    x_train,
    epochs=5,
    batch_size=128
)

random_points = np.random.normal(
    size=(5, latent_dim)
)

generated_images = decoder.predict(random_points)

plt.figure(figsize=(10, 2))

for i in range(5):
    plt.subplot(1, 5, i + 1)
    plt.imshow(
        generated_images[i].reshape(28, 28),
        cmap="gray"
    )
    plt.title(f"Image {i + 1}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
""" 
Task 5 — KL-Divergence Loss

# Prompt Used

Explain in simple beginner-friendly language what KL-divergence loss does in a Variational Autoencoder (VAE), why it is needed, and how it helps the VAE generate new images. Keep the answer within 3-4 sentences.

# ChatGPT Response

KL-divergence loss makes the latent space of a VAE follow a simple probability distribution, usually a standard normal distribution. It prevents the encoder from creating irregular and disconnected regions for different training examples. This makes the latent space smoother, so nearby points can represent similar images. Because the latent space is organized, we can randomly sample points from it and use the decoder to generate new images.

# What I learned

I learned that KL-divergence is not mainly used to reconstruct the image. Instead, it regularizes the latent space and makes it smooth and organized. This is important because it allows us to randomly sample latent vectors and generate new images.

"""